# Statystyka dla Data Analyst/Engineer — Moduł 2: Wizualizacja rozkładów

W Module 1 nauczyliśmy się opisywać dane liczbami: średnią, medianą, odchyleniem
standardowym, IQR. Te liczby są precyzyjne, ale potrafią ukryć kształt danych — dwa bardzo
różne zbiory mogą mieć niemal identyczne statystyki opisowe. W tym module nauczymy się
"zobaczyć" rozkład danych na wykresie, zamiast tylko go opisywać liczbami.

## Spis treści
1. [Dlaczego wizualizacja jest konieczna](#sec1)
2. [Histogram](#sec2)
3. [Wykres pudełkowy (boxplot)](#sec3)
4. [Wykres gęstości (KDE)](#sec4)
5. [Skośność i kurtoza](#sec5)
6. [Porównywanie rozkładów między grupami](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Dlaczego wizualizacja jest konieczna

Słynny przykład z lat 70. (tzw. **kwartet Anscombe'a**) pokazuje cztery zupełnie różne
zbiory danych, które mają niemal identyczną średnią, wariancję i korelację — a wyglądają
zupełnie inaczej na wykresie. Morał: **zawsze narysuj dane**, zanim zaufasz samym
statystykom opisowym. My zobaczymy to na własnych, prostszych przykładach.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


<a id="sec2"></a>
## 2. Histogram

**Histogram** dzieli zakres wartości na równe przedziały ("kubełki", *bins*) i pokazuje,
ile obserwacji wpada do każdego z nich. To najbardziej podstawowy sposób zobaczenia
kształtu rozkładu — czy jest symetryczny, skośny, jedno- czy wielomodalny.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(klienci["wydatki_miesieczne"], bins=30, edgecolor="white")
plt.xlabel("Wydatki miesieczne")
plt.ylabel("Liczba klientow")
plt.title("Histogram: wydatki miesieczne klientow")
plt.show()


Teraz widać wyraźnie to, co w Module 1 wywnioskowaliśmy tylko z liczb: rozkład ma długi
"ogon" w prawą stronę — większość klientów wydaje relatywnie mało, a garstka wydaje bardzo
dużo. To właśnie **skośność prawostronna**, o której będzie mowa w sekcji 5.

> ⚠️ **Liczba binów zmienia obraz**
>
> Zbyt mało binów spłaszcza histogram i ukrywa strukturę danych, zbyt dużo — sprawia, że wygląda na poszarpany szum. Nie ma jednej idealnej liczby; `bins=30` to rozsądny punkt startowy dla kilkuset-kilku tysięcy obserwacji, ale warto poeksperymentować (spróbuj podmienić na `bins=10` i `bins=100` w komórce powyżej i zobacz różnicę).

<a id="sec3"></a>
## 3. Wykres pudełkowy (boxplot)

**Boxplot** rysuje bezpośrednio to, co policzyliśmy ręcznie w Module 1: pudełko od Q1 do
Q3 (czyli IQR), linię w środku pudełka to mediana, "wąsy" sięgające do
`Q1 - 1.5*IQR`/`Q3 + 1.5*IQR`, a punkty poza wąsami to dokładnie te same outliery, które
liczyliśmy ręcznie.

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(klienci["wydatki_miesieczne"], vert=False)
plt.xlabel("Wydatki miesieczne")
plt.title("Boxplot: wydatki miesieczne klientow")
plt.show()


> 📊 **Boxplot to skompresowany Moduł 1**
>
> Jeśli rozumiesz boxplot, rozumiesz połowę Modułu 1 na jednym obrazku: mediana, Q1, Q3, IQR i outliery — wszystko na jednym, małym wykresie. To dlatego boxploty są tak popularne do szybkiego porównywania wielu grup naraz (zobaczysz to w sekcji 6).

<a id="sec4"></a>
## 4. Wykres gęstości (KDE)

**Wykres gęstości** (*Kernel Density Estimate*, KDE) to wygładzona wersja histogramu —
zamiast dyskretnych słupków rysuje ciągłą krzywą przybliżającą kształt rozkładu. Bywa
czytelniejszy niż histogram, zwłaszcza przy porównywaniu kilku rozkładów na jednym
wykresie.

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(klienci["wydatki_miesieczne"], fill=True)
plt.xlabel("Wydatki miesieczne")
plt.title("Wykres gestosci: wydatki miesieczne")
plt.show()


<a id="sec5"></a>
## 5. Skośność i kurtoza

Skoro "widzimy" skośność na oko, warto umieć ją też **zmierzyć liczbą**:

- **Skośność (skewness)** — mierzy asymetrię rozkładu. `0` to rozkład symetryczny,
  wartość dodatnia to skośność prawostronna (długi ogon w prawo, dokładnie jak nasze
  wydatki), wartość ujemna to skośność lewostronna.
- **Kurtoza (kurtosis)** — mierzy "grubość ogonów" rozkładu względem rozkładu normalnego.
  Wysoka kurtoza oznacza więcej ekstremalnych wartości (grubsze ogony) niż w rozkładzie
  normalnym.

In [ ]:
from scipy import stats

skosnosc = stats.skew(klienci["wydatki_miesieczne"])
kurtoza = stats.kurtosis(klienci["wydatki_miesieczne"])

print(f"Skosnosc: {skosnosc:.2f}")
print(f"Kurtoza: {kurtoza:.2f}")


> ⚠️ **scipy.stats.kurtosis zwraca 'nadmiarową' kurtozę**
>
> Domyślnie `scipy.stats.kurtosis` zwraca tzw. *excess kurtosis* — kurtozę pomniejszoną o 3, tak że rozkład normalny wychodzi jako `0` (a nie `3`, jak w podręcznikowej definicji). Dodatnia wartość = grubsze ogony niż rozkład normalny, ujemna = cieńsze.

<a id="sec6"></a>
## 6. Porównywanie rozkładów między grupami

Najczęstsze pytanie w praktyce brzmi nie "jaki jest rozkład X", tylko "czy rozkład X różni
się między grupami A i B?". Boxploty i histogramy świetnie się do tego nadają, gdy
narysujemy je obok siebie zamiast osobno.

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=klienci, x="region", y="satysfakcja")
plt.xlabel("Region")
plt.ylabel("Satysfakcja")
plt.title("Satysfakcja klientow wg regionu")
plt.show()


Już na oko widać, że mediana satysfakcji różni się między regionami (dokładnie ten
efekt celowo wbudowaliśmy w nasze dane). Czy ta różnica jest "prawdziwa", czy mogła
powstać przez przypadek? To pytanie wymaga już **testu statystycznego** — dokładnie do
tego typu pytania wrócimy w Module 10 (ANOVA), gdzie sprawdzimy tę samą różnicę regionów
formalnym testem.

In [ ]:
plt.figure(figsize=(8, 4))
for grupa, dane in klienci.groupby("grupa_kampanii"):
    sns.kdeplot(dane["wydatki_miesieczne"], label=f"Grupa {grupa}", fill=True, alpha=0.4)
plt.xlabel("Wydatki miesieczne")
plt.title("Wydatki wg grupy kampanii A/B")
plt.legend()
plt.show()


<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module nauczyliśmy się:
- rysować histogramy i interpretować kształt rozkładu,
- rysować boxploty i wiązać je z IQR/outlierami z Modułu 1,
- rysować wykresy gęstości (KDE) jako wygładzoną alternatywę dla histogramu,
- liczyć skośność i kurtozę jako liczbowe miary kształtu rozkładu,
- porównywać rozkłady między grupami wizualnie, jako wstęp do formalnych testów.

> 📝 **Ćwiczenie 1: Histogram wieku**
>
> Narysuj histogram kolumny `wiek` z 20 binami. Porównaj kształt z histogramem wydatków z sekcji 2 — czy wiek też jest skośny?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
plt.figure(figsize=(8, 4))
plt.hist(klienci["wiek"], bins=20, edgecolor="white")
plt.xlabel("Wiek")
plt.ylabel("Liczba klientow")
plt.title("Histogram: wiek klientow")
plt.show()
```

Wiek pochodzi z rozkladu jednostajnego, wiec histogram powinien wygladac w miare plasko/rownomiernie -- bez wyraznego ogona, w przeciwienstwie do wydatkow.
</details>

> 📝 **Ćwiczenie 2: Boxplot wg miasta**
>
> Narysuj boxplot `wydatki_miesieczne` osobno dla każdego `miasto` (jak w sekcji 6, ale dla miast zamiast regionów). Które miasto ma najwyższą medianę wydatków?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
plt.figure(figsize=(8, 4))
sns.boxplot(data=klienci, x="miasto", y="wydatki_miesieczne")
plt.xlabel("Miasto")
plt.ylabel("Wydatki miesieczne")
plt.title("Wydatki miesieczne wg miasta")
plt.show()
```
</details>

> 📝 **Ćwiczenie 3: Skośność satysfakcji**
>
> Policz skośność i kurtozę kolumny `satysfakcja`. Czy rozkład jest bardziej, czy mniej skośny niż wydatki z sekcji 5?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
from scipy import stats

print(stats.skew(klienci["satysfakcja"]))
print(stats.kurtosis(klienci["satysfakcja"]))
```

Satysfakcja jest budowana glownie z sumy wydatkow (przeskalowanych) i szumu normalnego, przycięta do zakresu 1-10 -- oczekuj duzo mniejszej skosnosci niz surowe wydatki.
</details>

> 📝 **Ćwiczenie 4: Wykres gęstości wg grupy**
>
> Narysuj jeden wykres z dwiema krzywymi gęstości (`sns.kdeplot`) kolumny `satysfakcja`, osobno dla `grupa_kampanii == 'A'` i `'B'`, na jednym wykresie z legendą.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
plt.figure(figsize=(8, 4))
for grupa, dane in klienci.groupby("grupa_kampanii"):
    sns.kdeplot(dane["satysfakcja"], label=f"Grupa {grupa}", fill=True, alpha=0.4)
plt.xlabel("Satysfakcja")
plt.title("Satysfakcja wg grupy kampanii")
plt.legend()
plt.show()
```
</details>

> 🔥 **Wyzwanie: siatka histogramów**
>
> Użyj `plt.subplots(2, 2, figsize=(10, 8))`, żeby narysować cztery histogramy (`wydatki_miesieczne`, `wiek`, `satysfakcja`, `wydatki_po`) w jednej siatce 2x2, każdy z własnym tytułem będącym nazwą kolumny. Podpowiedź: `axes.flat` pozwala iterować po wszystkich czterech osiach naraz.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
kolumny = ["wydatki_miesieczne", "wiek", "satysfakcja", "wydatki_po"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for kolumna, ax in zip(kolumny, axes.flat):
    ax.hist(klienci[kolumna], bins=20, edgecolor="white")
    ax.set_title(kolumna)
plt.tight_layout()
plt.show()
```
</details>

## Co dalej?

W **Module 3** cofniemy się do teorii: czym właściwie jest prawdopodobieństwo, jak liczyć
prawdopodobieństwo warunkowe i co mówi twierdzenie Bayesa. To fundament pod rozkłady
prawdopodobieństwa z Modułu 4 i pod całą resztę statystyki wnioskującej.